In [1]:
import os
os.environ["PROVIDER"] = "ollama"

In [2]:

from agent import run_agent

history = []  # история диалога: накапливается между ячейками, как в консольном чате

def ask(message: str) -> str:
    """Один ход диалога: печатает вопрос и ответ наставника."""
    answer = run_agent(message, history)
    history.append({"role": "user", "content": message})
    history.append({"role": "assistant", "content": answer})
    print(f"ВЫ: {message}\n\nНАСТАВНИК: {answer}")
    return answer

## Сценарий 1. Базовый диалог: студент просит условие задачи
Ожидание: агент вызовет `get_task` и перескажет условие `count_even`.

In [3]:
ask("Привет! Хочу решить задачу count_even. Расскажи условие.")

[вызов инструмента] get_task({'task_id': 'count_even'})
ВЫ: Привет! Хочу решить задачу count_even. Расскажи условие.

НАСТАВНИК: Задача "count_even" требует подсчитать количество чётных чисел в списке. Алгоритм:
1. Прочитать число N — количество элементов
2. Получить список из N целых чисел
3. Для каждого числа проверить чётность (остаток от деления на 2 равен 0)
4. Вывести общее количество таких чисел

Теперь напиши свой код — я проверю и помогу исправить ошибки.


'Задача "count_even" требует подсчитать количество чётных чисел в списке. Алгоритм:\n1. Прочитать число N — количество элементов\n2. Получить список из N целых чисел\n3. Для каждого числа проверить чётность (остаток от деления на 2 равен 0)\n4. Вывести общее количество таких чисел\n\nТеперь напиши свой код — я проверю и помогу исправить ошибки.'

## Сценарий 2. Код с типичной ошибкой (считает нечётные вместо чётных)
Ожидание: агент вызовет `run_tests`, увидит `wrong_answer` и даст подсказку БЕЗ готового кода.

In [4]:
ask("""Вот моё решение count_even, проверь:

```python
n = int(input())
nums = list(map(int, input().split()))
print(sum(1 for x in nums if x % 2 == 1))
```
""")

[вызов инструмента] run_tests({'code': 'n = int(input())\nnums = list(map(int, input().split()))\nprint(sum(1 for x in nums if x % 2 == 1))', 'task_id': 'count_even'})
[вызов инструмента] record_mistake({'task_id': 'count_even', 'category': 'logic', 'note': 'Фильтр отбирает нечётные числа, а нужно считать чётные'})
ВЫ: Вот моё решение count_even, проверь:

```python
n = int(input())
nums = list(map(int, input().split()))
print(sum(1 for x in nums if x % 2 == 1))
```


НАСТАВНИК: Фильтр отбирает нечётные числа — исправь условие, чтобы проверять чётность (остаток от деления на 2 должен быть 0). Попробуй снова.


'Фильтр отбирает нечётные числа — исправь условие, чтобы проверять чётность (остаток от деления на 2 должен быть 0). Попробуй снова.'

## Сценарий 3. Исправленное решение
Ожидание: все тесты пройдены; после разбора агент вызовет `record_mistake`
(или вызовет после следующей реплики — это нормально).

In [5]:
ask("""Понял, исправил:

```python
n = int(input())
nums = list(map(int, input().split()))
print(sum(1 for x in nums if x % 2 == 0))
```
""")

[вызов инструмента] run_tests({'task_id': 'count_even', 'code': 'n = int(input())\nnums = list(map(int, input().split()))\nprint(sum(1 for x in nums if x % 2 == 0))'})
ВЫ: Понял, исправил:

```python
n = int(input())
nums = list(map(int, input().split()))
print(sum(1 for x in nums if x % 2 == 0))
```


НАСТАВНИК: Ты молодец! Твой код прошёл все тесты. Проверяй дальше задачи по мере сил. Хочешь попробовать другую задачу? 😊


'Ты молодец! Твой код прошёл все тесты. Проверяй дальше задачи по мере сил. Хочешь попробовать другую задачу? 😊'

In [6]:
# Проверяем, что ошибка записалась в профиль студента (действие record_mistake)
import json, pathlib
p = pathlib.Path("student_profile.json")
print(json.dumps(json.loads(p.read_text(encoding="utf-8")), ensure_ascii=False, indent=2) if p.exists() else "Профиль пока пуст — спросите у наставника, что он записал о моих ошибках.")

{
  "mistakes": [
    {
      "task_id": "count_even",
      "category": "logic",
      "note": "Фильтр отбирает нечётные числа, а нужно считать чётные",
      "count": 4
    }
  ]
}


## Сценарий 4. Обработка ошибок: несуществующая задача и бесконечный цикл
Ожидание: инструмент возвращает ошибку, агент честно сообщает о ней и НЕ выдумывает ответ.

In [7]:
ask("Дай условие задачи bubble_sort.")

ВЫ: Дай условие задачи bubble_sort.

НАСТАВНИК: Задачи с названием "bubble_sort" в текущем каталоге нет. Доступны задачи: sum_two, count_even, max_of_three. Проверь название или выбери другую задачу — я помогу! 😊


'Задачи с названием "bubble_sort" в текущем каталоге нет. Доступны задачи: sum_two, count_even, max_of_three. Проверь название или выбери другую задачу — я помогу! 😊'

In [8]:
ask("""Проверь ещё вариант count_even:

```python
while True:
    pass
```
""")

[вызов инструмента] run_tests({'code': 'while True:\\n    pass', 'task_id': 'count_even'})
ВЫ: Проверь ещё вариант count_even:

```python
while True:
    pass
```


НАСТАВНИК: Твой код не обрабатывает данные — цикл while True без логики внутри не работает. Опиши, как ты думаешь, нужно считать чётные числа? Я помогу найти ошибку. 😊


'Твой код не обрабатывает данные — цикл while True без логики внутри не работает. Опиши, как ты думаешь, нужно считать чётные числа? Я помогу найти ошибку. 😊'

## Сценарий 5. Смена провайдера без смены кода
Меняем только переменные окружения и повторяем сценарий 2 — поведение должно быть сопоставимым.
После смены ячейки ниже перезапускать ядро НЕ нужно: клиент создаётся на каждый диалог.

In [10]:
print("Текущий провайдер:", os.environ.get("PROVIDER"))

Текущий провайдер: ollama
